In [ ]:
import os
import json
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료!")

In [ ]:
# klec txt 파일 전체 읽기
txt_files = glob.glob('../data/klec/**/*.txt', recursive=True)
print(f"총 txt 파일 수: {len(txt_files)}")

# 파일 몇 개 내용 확인
for f in txt_files[:3]:
    with open(f, 'r', encoding='utf-8') as file:
        content = file.read().strip()
    print(f"\n파일: {os.path.basename(f)}")
    print(f"내용: {content}")

In [ ]:
# 필러워드 태그 정의
FILLER_TAGS = ['아/', '어/', '음/', '그/', '뭐/', '저/', '이/', 'n/']

def extract_features(text):
    words = text.strip().split()
    total = len(words)
    if total == 0:
        return None

    # 필러워드 수
    filler_count = sum(1 for w in words if any(w.startswith(tag) or w == tag[:-1] for tag in FILLER_TAGS))

    # 수정: 한글 태그까지 제거하는 정규식
    # FILLER_TAGS 패턴을 직접 단어 단위로 필터링
    clean_words = [w for w in words if not any(w.startswith(tag) or w == tag[:-1] for tag in FILLER_TAGS)]
    
    # 추가로 슬래시 포함 단어도 제거 (혹시 모를 태그 잔여물)
    clean_words = [w for w in clean_words if '/' not in w]
    
    # 추가: (N번), 번), (5번) 같은 발화 순서 표기 제거
    clean_words = [w for w in clean_words if not re.match(r'^\(?\d*번\)?$', w)]

    clean_words = [w for w in clean_words if w.strip()]

    # 어휘 다양성
    vocab_diversity = len(set(clean_words)) / len(clean_words) if clean_words else 0

    # 평균 단어 길이
    avg_word_len = np.mean([len(w) for w in clean_words]) if clean_words else 0

    return {
        'total_words': total,
        'filler_count': filler_count,
        'filler_ratio': filler_count / total,
        'vocab_diversity': vocab_diversity,
        'avg_word_len': avg_word_len,
        'text': text.strip(),
        'clean_text': ' '.join(clean_words)   # 태그 완전 제거된 텍스트
    }

In [ ]:
# 전체 txt 파일에서 feature 추출 (샘플 5000개만)
import random
random.seed(42)

sampled_files = random.sample(txt_files, min(10000, len(txt_files)))
print(f"샘플 파일 수: {len(sampled_files)}")

records = []
for filepath in sampled_files:
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read().strip()
        
        if len(text) < 2:  # 너무 짧은 텍스트 제외
            continue
            
        features = extract_features(text)
        if features:
            records.append(features)
    except:
        continue

df = pd.DataFrame(records)
print(f"\n추출 완료! 총 {len(df)}개")
print(df.describe())

In [ ]:
# 레이블 기준 (EDA에서 확인한 값 기반)
# 필러워드 비율 > 17% → Poor (0)
# 필러워드 비율 <= 17% → Good (1)

df['label'] = (df['filler_ratio'] <= 0.17).astype(int)

print("=== 레이블 분포 ===")
print(df['label'].value_counts())
print(f"\nGood 비율: {df['label'].mean()*100:.1f}%")
print(f"Poor 비율: {(1-df['label'].mean())*100:.1f}%")

# 레이블 분포 시각화
plt.figure(figsize=(6,4))
df['label'].value_counts().plot(kind='bar', color=['steelblue', 'coral'])
plt.xticks([0, 1], ['Good (1)', 'Poor (0)'], rotation=0)
plt.title('레이블 분포')
plt.ylabel('빈도')
plt.savefig('../results/label_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("그래프 저장 완료!")

In [ ]:
# 기준 조정 — 중앙값(8%) 기준으로 나누기
df['label'] = (df['filler_ratio'] <= df['filler_ratio'].median()).astype(int)

print("=== 조정된 레이블 분포 ===")
print(df['label'].value_counts())
print(f"\nGood 비율: {df['label'].mean()*100:.1f}%")
print(f"Poor 비율: {(1-df['label'].mean())*100:.1f}%")

colab 학습 루프에서 데이터 누수 발생으로 아래 코드 변경. text 추가

In [ ]:
# feature 컬럼만 선택
feature_cols = ['total_words', 'filler_count', 'filler_ratio', 
                'vocab_diversity', 'avg_word_len']

X = df[feature_cols]
y = df['label']
text_col = df['text']
clean_text_col = df['clean_text']  # ← 추가

# Train/Test 분리 (8:2)
X_train, X_test, y_train, y_test, text_train, text_test, clean_train, clean_test = train_test_split(
    X, y, text_col, clean_text_col, test_size=0.2, random_state=42, stratify=y)

# Train/Val 분리 (Train의 8:2 → 전체 7:1:2)
X_train, X_val, y_train, y_val, text_train, text_val, clean_train, clean_val = train_test_split(
    X_train, y_train, text_train, clean_train, test_size=0.125, random_state=42, stratify=y_train)

print(f"Train: {len(X_train)}개")
print(f"Val:   {len(X_val)}개")
print(f"Test:  {len(X_test)}개")

# CSV 저장
train_df = X_train.copy()
train_df['label'] = y_train
train_df['text'] = text_train.values
train_df['clean_text'] = clean_train.values
val_df = X_val.copy()
val_df['label'] = y_val
val_df['text'] = text_val.values
val_df['clean_text'] = clean_val.values
test_df = X_test.copy()
test_df['label'] = y_test
test_df['text'] = text_test.values
test_df['clean_text'] = clean_test.values

train_df.to_csv('../data/train.csv', index=False)
val_df.to_csv('../data/val.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)

print("\nCSV 저장 완료!")
print(f"컬럼: {list(train_df.columns)}")
print(train_df['clean_text'].head(3))

In [ ]:
sample_text = "n/ 아/ 큰일 났다, 우리교실 어디있지?"
result = extract_features(sample_text)
print(result['clean_text'])
# 기대 출력: "큰일 났다, 우리교실 어디있지?"